<a href="https://colab.research.google.com/github/sjmn-zip/Daon/blob/main/Daon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# 드라이브 경로로 불러오기
path = "/content/drive/MyDrive/Daon/감성대화말뭉치(최종데이터)_Training.json"
df = pd.read_json(path)

print(df.shape)
print(df.head())

(51628, 2)
                                             profile  \
0  {'persona-id': 'Pro_05349', 'persona': {'perso...   
1  {'persona-id': 'Pro_05349', 'persona': {'perso...   
2  {'persona-id': 'Pro_05349', 'persona': {'perso...   
3  {'persona-id': 'Pro_05349', 'persona': {'perso...   
4  {'persona-id': 'Pro_05349', 'persona': {'perso...   

                                                talk  
0  {'id': {'profile-id': 'Pro_05349', 'talk-id': ...  
1  {'id': {'profile-id': 'Pro_05349', 'talk-id': ...  
2  {'id': {'profile-id': 'Pro_05349', 'talk-id': ...  
3  {'id': {'profile-id': 'Pro_05349', 'talk-id': ...  
4  {'id': {'profile-id': 'Pro_05349', 'talk-id': ...  


In [ ]:
# 감정 코드 → 감정 이름 변환 함수 (Phase 2에서 만든 것)
def code_to_emotion(emotion_code):
    code_num = emotion_code[1]
    if code_num == "1": return "분노"
    elif code_num == "2": return "슬픔"
    elif code_num == "3": return "불안"
    elif code_num == "4": return "상처"
    elif code_num == "5": return "당황"
    elif code_num == "6": return "기쁨"
    else: return "알 수 없음"

# 5만 개 전체에서 문장 + 감정 뽑기
sentences = []
emotions = []
for i in range(len(df)):
    talk = df.loc[i, "talk"]
    profile = df.loc[i, "profile"]
    sentences.append(talk["content"]["HS01"])
    emotions.append(code_to_emotion(profile["emotion"]["type"]))

data = pd.DataFrame({"sentence": sentences, "emotion": emotions})

print(data.shape)
print(data.head())
print(data["emotion"].value_counts())

(51628, 2)
                                            sentence emotion
0                          일은 왜 해도 해도 끝이 없을까? 화가 난다.      분노
1     이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나.      분노
2  회사에 신입이 들어왔는데 말투가 거슬려. 그런 애를 매일 봐야 한다고 생각하니까 스...      분노
3  직장에서 막내라는 이유로 나에게만 온갖 심부름을 시켜. 일도 많은 데 정말 분하고 ...      분노
4              얼마 전 입사한 신입사원이 나를 무시하는 것 같아서 너무 화가 나.      분노
emotion
불안    9319
분노    9160
상처    9142
슬픔    9125
당황    8756
기쁨    6126
Name: count, dtype: int64


In [ ]:
# 감정 이름 → 숫자 매핑
label_map = {"분노": 0, "슬픔": 1, "불안": 2, "상처": 3, "당황": 4, "기쁨": 5}

# emotion 열을 숫자로 바꿔서 새 열(label) 추가
data["label"] = data["emotion"].map(label_map)

print(data.head())
print(data["label"].value_counts())

                                            sentence emotion  label
0                          일은 왜 해도 해도 끝이 없을까? 화가 난다.      분노      0
1     이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나.      분노      0
2  회사에 신입이 들어왔는데 말투가 거슬려. 그런 애를 매일 봐야 한다고 생각하니까 스...      분노      0
3  직장에서 막내라는 이유로 나에게만 온갖 심부름을 시켜. 일도 많은 데 정말 분하고 ...      분노      0
4              얼마 전 입사한 신입사원이 나를 무시하는 것 같아서 너무 화가 나.      분노      0
label
2    9319
0    9160
3    9142
1    9125
4    8756
5    6126
Name: count, dtype: int64


In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoTokenizer

# KoBERT 토크나이저 가져오기
tokenizer = AutoTokenizer.from_pretrained("monologg/kobert", trust_remote_code=True)

# 테스트: 문장 하나 토큰화해보기
sample = "요즘 너무 힘들고 지쳐요"
result = tokenizer(sample)
print(result)

tokenizer_config.json:   0%|          | 0.00/263 [00:00<?, ?B/s]

tokenization_kobert.py:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/monologg/kobert:
- tokenization_kobert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_78b3253a26.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/77.8k [00:00<?, ?B/s]

{'input_ids': [2, 3489, 1458, 5213, 5439, 4297, 7443, 6999, 3], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
from sklearn.model_selection import train_test_split

# 문장(X)과 정답 라벨(y) 분리
X = data["sentence"].tolist()   # 문장들을 리스트로
y = data["label"].tolist()      # 감정 숫자들을 리스트로

# 8:2로 학습/검증 나누기
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("학습용:", len(X_train))
print("검증용:", len(X_test))

학습용: 41302
검증용: 10326


In [ ]:
# 학습용 문장 전체를 토큰화
train_encodings = tokenizer(
    X_train,
    truncation=True,
    padding=True,
    max_length=64,
    return_tensors="pt"
)

# 검증용 문장도 똑같이
test_encodings = tokenizer(
    X_test,
    truncation=True,
    padding=True,
    max_length=64,
    return_tensors="pt"
)

print(train_encodings["input_ids"].shape)

torch.Size([41302, 64])


In [ ]:
import torch

# 문장 + 정답을 하나로 묶는 상자(Dataset) 정의
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# 학습용 / 검증용 상자 만들기
train_dataset = EmotionDataset(train_encodings, y_train)
test_dataset = EmotionDataset(test_encodings, y_test)

print("학습 데이터셋 크기:", len(train_dataset))
print("검증 데이터셋 크기:", len(test_dataset))

학습 데이터셋 크기: 41302
검증 데이터셋 크기: 10326


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "monologg/kobert",
    num_labels=6,
    trust_remote_code=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: monologg/kobert
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import Trainer, TrainingArguments

# 학습 설정
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    logging_steps=100,
)

# Trainer 만들기
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.232886,1.167374
2,1.029554,1.134552


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5164, training_loss=1.167836343660362, metrics={'train_runtime': 1138.9809, 'train_samples_per_second': 72.524, 'train_steps_per_second': 4.534, 'total_flos': 2716850772642816.0, 'train_loss': 1.167836343660362, 'epoch': 2.0})

In [ ]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch
1.029554,1.134552,2


{'eval_loss': 1.134552240371704}


In [ ]:
import numpy as np

# 검증 데이터로 예측 뽑기
predictions = trainer.predict(test_dataset)

# 예측값 (6개 중 가장 확률 높은 감정 고르기)
preds = np.argmax(predictions.predictions, axis=1)

# 정답
labels = predictions.label_ids

# 정확도 = 맞힌 개수 / 전체
accuracy = (preds == labels).mean()
print(f"정확도: {accuracy * 100:.2f}%")

정확도: 57.40%
